In [ ]:
import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D,
    LSTM, Dense
)
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split


In [ ]:
LABEL_MAP = {
    "_corr": 0,
    "_id": 1,
    "_rb": 2,
    "_toe": 3
}
NUM_CLASSES = 4

DATA_ROOT = "..."

X, y = [], []

for root, _, files in os.walk(DATA_ROOT):
    for file in files:
        if file.endswith(".npy"):
            for suffix, label_id in LABEL_MAP.items():
                if suffix in root:
                    data = np.load(os.path.join(root, file))
                    data = data.reshape(50, 34)
                    X.append(data)
                    y.append(label_id)
                    break

X = np.array(X)
y = to_categorical(y, NUM_CLASSES)

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y.argmax(axis=1)
)


In [ ]:
input_layer = Input(shape=(50, 34))

x = Conv1D(32, kernel_size=3, activation="relu")(input_layer)
x = MaxPooling1D(pool_size=2)(x)
x = LSTM(64)(x)
x = Dense(32, activation="relu")(x)

output = Dense(4, activation="softmax")(x)

model_cnn_lstm = Model(input_layer, output)


In [ ]:
model_cnn_lstm.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model_cnn_lstm.summary()


In [ ]:
history_cnn_lstm = model_cnn_lstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=16,
    verbose=1
)


In [ ]:
model_cnn_lstm.save(
    "1D_CNN_Posture_LSTM.keras"
)
